In [0]:
import pandas as pd
import requests

# ------------------------------------------------------------------
# 1) MUNICÍPIOS (IBGE) + UF/REGIÃO (igual já tínhamos)
# ------------------------------------------------------------------
municipios = pd.read_csv("https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv")
municipios["codigo_municipio"] = municipios["codigo_ibge"] // 10  # remove o dígito verificador

dim_uf = pd.DataFrame([
    (11, "RO", "Rondônia", "Norte"), (12, "AC", "Acre", "Norte"), (13, "AM", "Amazonas", "Norte"),
    (14, "RR", "Roraima", "Norte"), (15, "PA", "Pará", "Norte"), (16, "AP", "Amapá", "Norte"), (17, "TO", "Tocantins", "Norte"),
    (21, "MA", "Maranhão", "Nordeste"), (22, "PI", "Piauí", "Nordeste"), (23, "CE", "Ceará", "Nordeste"),
    (24, "RN", "Rio Grande do Norte", "Nordeste"), (25, "PB", "Paraíba", "Nordeste"), (26, "PE", "Pernambuco", "Nordeste"),
    (27, "AL", "Alagoas", "Nordeste"), (28, "SE", "Sergipe", "Nordeste"), (29, "BA", "Bahia", "Nordeste"),
    (31, "MG", "Minas Gerais", "Sudeste"), (32, "ES", "Espírito Santo", "Sudeste"), (33, "RJ", "Rio de Janeiro", "Sudeste"),
    (35, "SP", "São Paulo", "Sudeste"),
    (41, "PR", "Paraná", "Sul"), (42, "SC", "Santa Catarina", "Sul"), (43, "RS", "Rio Grande do Sul", "Sul"),
    (50, "MS", "Mato Grosso do Sul", "Centro-Oeste"), (51, "MT", "Mato Grosso", "Centro-Oeste"),
    (52, "GO", "Goiás", "Centro-Oeste"), (53, "DF", "Distrito Federal", "Centro-Oeste"),
], columns=["codigo_uf", "sigla_uf", "nome_uf", "regiao"])

dim_municipio = municipios.merge(dim_uf, on="codigo_uf")[
    ["codigo_municipio", "nome", "codigo_uf", "sigla_uf", "nome_uf", "regiao", "latitude", "longitude"]
].rename(columns={"nome": "nome_municipio"})

ignorados = dim_uf.copy()
ignorados["codigo_municipio"] = ignorados["codigo_uf"] * 10000
ignorados["nome_municipio"] = "Ignorado (" + ignorados["sigla_uf"] + ")"
ignorados["latitude"] = None
ignorados["longitude"] = None
ignorados = ignorados[["codigo_municipio", "nome_municipio", "codigo_uf", "sigla_uf", "nome_uf", "regiao", "latitude", "longitude"]]

dim_municipio_final = pd.concat([dim_municipio, ignorados], ignore_index=True)
print("Total de linhas em dim_municipio (esperado: 5.571 + 27 = 5.598):", len(dim_municipio_final))

# ------------------------------------------------------------------
# 2) POPULAÇÃO POR COR/RAÇA (Censo 2022) — NOVO
#    Fonte fixa (só 2022, não existe série anual por raça).
#    Vira atributo da dimensão, não tabela separada, porque não varia por ano no nosso dataset.
# ------------------------------------------------------------------
url_raca = "https://apisidra.ibge.gov.br/values/t/9605/n6/all/v/93/p/2022/c86/all"
resp_raca = requests.get(url_raca, timeout=120)
print("\nStatus HTTP (população por raça):", resp_raca.status_code)

dados_raca = resp_raca.json()[1:]  # remove cabeçalho
df_raca_long = pd.DataFrame(dados_raca)[["D1C", "D4N", "V"]]
df_raca_long.columns = ["codigo_ibge", "cor_raca", "populacao"]
df_raca_long["codigo_municipio"] = df_raca_long["codigo_ibge"].astype(int) // 10
df_raca_long["populacao"] = pd.to_numeric(df_raca_long["populacao"], errors="coerce")

# Pivota de formato longo (1 linha por município+raça) pra largo (1 linha por município, 1 coluna por raça)
df_raca_wide = df_raca_long.pivot_table(
    index="codigo_municipio",
    columns="cor_raca",
    values="populacao",
    aggfunc="first"
).reset_index()

df_raca_wide = df_raca_wide.rename(columns={
    "Total": "populacao_total_censo2022",
    "Branca": "populacao_branca_censo2022",
    "Preta": "populacao_preta_censo2022",
    "Amarela": "populacao_amarela_censo2022",
    "Parda": "populacao_parda_censo2022",
    "Indígena": "populacao_indigena_censo2022",
})

print("Municípios com população por raça (esperado 5.570, falta Boa Esperança do Norte):", len(df_raca_wide))

# ------------------------------------------------------------------
# 3) JUNTAR COM A DIM_MUNICIPIO (left join: quem não tem dado de raça
#    fica com essas colunas nulas — Boa Esperança do Norte e as 27
#    linhas "Ignorado" sintéticas, que nunca tiveram Censo de verdade)
# ------------------------------------------------------------------
dim_municipio_final = dim_municipio_final.merge(df_raca_wide, on="codigo_municipio", how="left")

qtd_nulos = dim_municipio_final["populacao_total_censo2022"].isna().sum()
print("Linhas sem dado de raça (esperado 28: 1 município real + 27 'Ignorado'):", qtd_nulos)

# ------------------------------------------------------------------
# 4) GRAVAR
# ------------------------------------------------------------------
spark_dim_municipio = spark.createDataFrame(dim_municipio_final)
(spark_dim_municipio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_dim_municipio"))

print("\nTabela 'gold_dim_municipio' gravada com as colunas de raça/cor.")

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# 1) BUSCAR 2015-2021 e 2024 NA TABELA 6579 (estimativas oficiais)
# ------------------------------------------------------------------
url_6579 = "https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/9324/p/2015-2024"
resp_6579 = requests.get(url_6579, timeout=120)
print("Status HTTP (6579):", resp_6579.status_code)

dados_6579 = resp_6579.json()[1:]  # remove o cabeçalho
df_6579 = pd.DataFrame(dados_6579)[["D1C", "D3C", "V"]]
df_6579.columns = ["codigo_ibge", "ano", "populacao"]
df_6579["codigo_municipio"] = df_6579["codigo_ibge"].astype(int) // 10
df_6579["ano"] = df_6579["ano"].astype(int)
# errors="coerce": valores tipo "..." (código do SIDRA pra "não disponível")
# viram NaN em vez de quebrar o script
df_6579["populacao"] = pd.to_numeric(df_6579["populacao"], errors="coerce")
df_6579["populacao_estimada"] = False
df_6579 = df_6579[["ano", "codigo_municipio", "populacao", "populacao_estimada"]]

qtd_nulos_6579 = df_6579["populacao"].isna().sum()
print("Linhas com população nula na 6579 (esperado 8, o Boa Esperança do Norte):", qtd_nulos_6579)
print("Anos vindos da tabela 6579:", sorted(df_6579["ano"].unique()))
print("Linhas:", len(df_6579))

# ------------------------------------------------------------------
# 2) BUSCAR 2022 NA TABELA 9514 (Censo 2022)
# ------------------------------------------------------------------
url_9514 = "https://apisidra.ibge.gov.br/values/t/9514/n6/all/v/allxp/p/2022"
resp_9514 = requests.get(url_9514, timeout=120)
print("\nStatus HTTP (9514):", resp_9514.status_code)

dados_9514 = resp_9514.json()[1:]  # remove o cabeçalho
df_9514 = pd.DataFrame(dados_9514)[["D1C", "D3C", "V"]]
df_9514.columns = ["codigo_ibge", "ano", "populacao"]
df_9514["codigo_municipio"] = df_9514["codigo_ibge"].astype(int) // 10
df_9514["ano"] = df_9514["ano"].astype(int)
df_9514["populacao"] = pd.to_numeric(df_9514["populacao"], errors="coerce")
df_9514["populacao_estimada"] = False
df_9514 = df_9514[["ano", "codigo_municipio", "populacao", "populacao_estimada"]]

print("Municípios em 2022 (esperado 5.570, falta Boa Esperança do Norte):", len(df_9514))

# ------------------------------------------------------------------
# 3) BOA ESPERANÇA DO NORTE (510183): não existe dado em NENHUMA fonte
#    testada (nem 6579, nem 9514). Em vez de inventar um número sem
#    base, adicionamos ele explicitamente com população NULA em 2022,
#    documentando a limitação em vez de escondê-la.
# ------------------------------------------------------------------
codigo_sem_dado = 510183
if codigo_sem_dado not in df_9514["codigo_municipio"].values:
    linha_sem_dado = pd.DataFrame([{
        "ano": 2022,
        "codigo_municipio": codigo_sem_dado,
        "populacao": None,
        "populacao_estimada": True,  # marcado como não confiável / indisponível
    }])
    df_9514 = pd.concat([df_9514, linha_sem_dado], ignore_index=True)
    print(f"Município {codigo_sem_dado} adicionado em 2022 com população NULA (sem fonte disponível).")

print("Municípios em 2022 depois do ajuste (esperado 5.571):", len(df_9514))

# ------------------------------------------------------------------
# 4) CRIAR 2023 POR INTERPOLAÇÃO LINEAR ENTRE 2022 E 2024
#    (onde faltar 2022 ou 2024 -- caso do Boa Esperança do Norte --
#    o resultado da interpolação também fica nulo, propagando a
#    limitação de forma consistente em vez de mascará-la)
# ------------------------------------------------------------------
pop_2022 = df_9514[["codigo_municipio", "populacao"]].rename(columns={"populacao": "pop_2022"})
pop_2024 = df_6579[df_6579["ano"] == 2024][["codigo_municipio", "populacao"]].rename(columns={"populacao": "pop_2024"})

interp = pop_2022.merge(pop_2024, on="codigo_municipio", how="inner")
interp["populacao"] = ((interp["pop_2022"] + interp["pop_2024"]) / 2).round()
interp["ano"] = 2023
interp["populacao_estimada"] = True
df_2023 = interp[["ano", "codigo_municipio", "populacao", "populacao_estimada"]]

print("\nMunicípios com 2023 (esperado 5.571):", len(df_2023))
print("População nula em 2023 (esperado 1):", df_2023["populacao"].isna().sum())

# ------------------------------------------------------------------
# 5) JUNTAR TUDO (2015-2021, 2022, 2023, 2024) E GRAVAR
# ------------------------------------------------------------------
df_populacao_final = pd.concat([df_6579, df_9514, df_2023], ignore_index=True)
print("\nTotal de linhas final (esperado: 5.571 municípios x 10 anos = 55.710):", len(df_populacao_final))
print("Anos presentes:", sorted(df_populacao_final["ano"].unique()))
print("Linhas com população nula no total (esperado 10, 1 por ano, o Boa Esperança do Norte):",
      df_populacao_final["populacao"].isna().sum())

spark_populacao = spark.createDataFrame(df_populacao_final)
(spark_populacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_populacao_municipio"))

print("Tabela 'gold_populacao_municipio' gravada com sucesso.")

In [0]:
import requests

# Testa para TODOS os municípios, só ano de 2022 (Censo)
url = "https://apisidra.ibge.gov.br/values/t/9605/n6/all/v/93/p/2022/c86/all"

resp = requests.get(url, timeout=120)
print("Status HTTP:", resp.status_code)

if resp.status_code == 200:
    dados = resp.json()
    # Esperado: 1 cabeçalho + (5.571 municípios x 6 categorias [Total + 5 raças]) = 33.427
    print("Quantidade de registros retornados (esperado: 33.427):", len(dados))

    municipios = set(linha["D1C"] for linha in dados[1:])
    print("Municípios distintos (esperado 5.570 ou 5.571):", len(municipios))

    categorias = set(linha["D4N"] for linha in dados[1:])
    print("Categorias de cor/raça encontradas:", categorias)
else:
    print("Resposta (texto bruto):", resp.text)